In [7]:
import requests
import tarfile
import gzip
import shutil
#!pip install osgeo
from osgeo import gdal
import calendar
from datetime import datetime

In [1]:
#date = "20250331" # for manual date setting
date = datetime.today().strftime('%Y%m%d') # get today's date
print(date)

NameError: name 'datetime' is not defined

In [9]:
year = date[0:4]
month = date[4:6]
month_abbrev = calendar.month_abbr[int(month)]
file_path = "https://noaadata.apps.nsidc.org/NOAA/G02158/masked/" + year + "/" + month + "_" + month_abbrev + "/"
file_name = "SNODAS_" + date + ".tar"
file_url = file_path + file_name

## download file from SNODAS website
r = requests.get(file_url)

'''
r.raise_for_status()
except requests.exceptions.HTTPError as err:
    raise SystemExit(err)       
    '''

with open("../data/SNODAS/" + file_name, "wb") as f:	
    f.write(r.content)
    
## untar file
tar = tarfile.open("../data/SNODAS/" + file_name)
tar.extractall(path="../data/SNODAS/")
tar.close()

## unzip SWE dat file
with gzip.open("../data/SNODAS/us_ssmv11034tS__T0001TTNATS" + date + "05HP001.dat.gz", 'rb') as file_in:
    with open("../data/SNODAS/us_ssmv11034tS__T0001TTNATS" + date + "05HP001.dat", 'wb') as file_out:
        shutil.copyfileobj(file_in, file_out)

## create hdr file 
hdr_content = """ENVI
samples = 6935
lines = 3351
bands = 1
header offset = 0
file type = ENVI Standard
data type = 2
interleave = bsq
byte order = 1
"""

## save .hdr file
hdr_filename = "../data/SNODAS/us_ssmv11034tS__T0001TTNATS" + date + "05HP001.hdr"
with open(hdr_filename, "w") as f:
    f.write(hdr_content)

## convert .dat file to .tif with gdal_translate
input_file = "../data/SNODAS/us_ssmv11034tS__T0001TTNATS" + date + "05HP001.dat"
#output_file = "SNODAS_Data/us_ssmv11034tS__T0001TTNATS" + date + "05HP001.tif"
output_file = "../data/SNODAS/SNODAS_SWE_" +  date + ".tif"

## open input file
dataset = gdal.Open(input_file, gdal.GA_ReadOnly)

# Set output format and options
output_format = "GTiff"  # GeoTIFF format
options = [
    "-a_srs", "+proj=longlat +ellps=WGS84 +datum=WGS84 +no_defs",  # Set spatial reference
    "-a_nodata", "-9999",  # Set NoData value
    "-a_ullr", "-124.73333333333333", "52.87500000000000", "-66.94166666666667", "24.95000000000000"  # Set bounding box
]

# Perform translation (conversion)
gdal.Translate(output_file, dataset, format=output_format, options=options)

# Close dataset
dataset = None

In [12]:
file_url

'https://noaadata.apps.nsidc.org/NOAA/G02158/masked/2025/04_Apr/SNODAS_20250402.tar'